# A typed FastAPI endpoint around a model call

A garden-centre chain moves its plant-care question bot from a developer's laptop script to a web service that
the shop tablets call. In this notebook you check the five real tablet requests by hand in plain Python, then
build the same contract with FastAPI and Pydantic: a request model that answers 422 before any model call, a
response model that filters what leaves, and a clean 502 when the model cannot be reached.

About 10 minutes. Run the cells from top to bottom. Everything runs on a free CPU runtime, with no account and no key.
The model answers are the lesson's recorded answers, replayed. No server is started: FastAPI's `TestClient`
sends each request to the app inside this notebook.

## 1. Install the versions the lesson ran

The next cell installs FastAPI 0.162.6 with Starlette 1.10.0, Pydantic 2.14.3 and httpx 0.28.1.

**Look for:** no error. pip may print a few lines about other packages; you can ignore them.

In [1]:
%pip install -q fastapi==0.162.6 starlette==1.10.0 pydantic==2.14.3 httpx==0.28.1

Note: you may need to restart the kernel to use updated packages.


## 2. The five tablet requests

These are the five request bodies the tablets sent, as Python dicts (the tablet sends JSON; Python reads it as a
dict). The contract the two teams agreed on: `question` is required, text, 1 to 300 characters; `plant` is
optional text; anything else is not used.

**Look for:** request 2 (`fern-old-app`) has `text` and no `question`. Request 4 (`rose-long`) has a question of
**386** characters. Request 5 (`tomato-extra`) has an extra field, `tablet_id`.

In [2]:
import json

tablet_requests = [
    {"name": "basil", "body": {"plant": "basil", "question": "The lower leaves on my basil are turning yellow. What should I do?"}},
    {"name": "fern-old-app", "body": {"plant": "fern", "text": "Can I keep a fern in a bathroom with no window?"}},
    {"name": "cactus", "body": {"question": "How often should I water a cactus in winter?"}},
    {"name": "rose-long", "body": {"plant": "rose", "question": "My rose bush was planted three years ago by the previous owner of the house, next to a wooden fence that gets sun in the morning and shade after lunch. This spring the leaves got black spots, then turned yellow and fell off, first at the bottom and then higher up. I sprayed it with something from the shop but I do not remember the name. What is wrong with it and what should I do now?"}},
    {"name": "tomato-extra", "body": {"plant": "tomato", "question": "Why do my tomato flowers fall off before fruit forms?", "tablet_id": "shop-12-tablet-3"}},
]

for i, r in enumerate(tablet_requests, 1):
    q = r["body"].get("question")
    print(f"request {i} {r['name']:<13} keys: {list(r['body'])}  question length: {len(q) if isinstance(q, str) else None}")

request 1 basil         keys: ['plant', 'question']  question length: 66
request 2 fern-old-app  keys: ['plant', 'text']  question length: None
request 3 cactus        keys: ['question']  question length: 44
request 4 rose-long     keys: ['plant', 'question']  question length: 386
request 5 tomato-extra  keys: ['plant', 'question', 'tablet_id']  question length: 53


## 3. The check, by hand in plain Python

Before any framework: the contract as a plain function, the same 10 lines as on the lesson's screen. It returns
a status code and either the errors or a clean copy of the request. (The check that plant is text is left out to
keep it short.)

**Look for:** 200, 422, 200, 422, 200. Request 2 fails with `missing or not text`: `body.get("question")` gave
None, and `text` was never read. Request 4 fails with `must be 1 to 300 characters`. The tomato copy has only
plant and question: `tablet_id` is gone.

In [3]:
MAX_QUESTION = 300

def check_ask(body):
    errors = []
    question = body.get("question")
    if not isinstance(question, str):
        errors.append(["question", "missing or not text"])
    elif not 1 <= len(question) <= MAX_QUESTION:
        errors.append(["question", "must be 1 to 300 characters"])
    if errors:
        return 422, errors
    return 200, {"plant": body.get("plant"), "question": question}

for r in tablet_requests:
    print(r["name"], "->", json.dumps(check_ask(r["body"])))

basil -> [200, {"plant": "basil", "question": "The lower leaves on my basil are turning yellow. What should I do?"}]
fern-old-app -> [422, [["question", "missing or not text"]]]
cactus -> [200, {"plant": null, "question": "How often should I water a cactus in winter?"}]
rose-long -> [422, [["question", "must be 1 to 300 characters"]]]
tomato-extra -> [200, {"plant": "tomato", "question": "Why do my tomato flowers fall off before fruit forms?"}]


## 4. The recorded model answers, replayed

The lesson asked a small hosted model (google/gemini-2.5-flash-lite, temperature 0, on 2026-11-13) the three
questions that pass the contract. You have no API key, so `call_model` replays those recorded answers: it builds
the same user message the recordings were made with and looks up the answer. Every call is added to `calls`,
so you can count model calls.

**Look for:** three recorded user messages. The cactus request has no plant, so its message starts with
`Question:` only.

In [4]:
SYSTEM = "You answer plant-care questions from customers at a garden centre's shop tablet. Answer in at most three short sentences, in plain words. If the question is not about plants or gardening, say that you can only help with plants."

RECORDED = {  # copied from the lesson's recordings: the user message and the model's answer
    "ask-basil": {
        "user": "Plant: basil. Question: The lower leaves on my basil are turning yellow. What should I do?",
        "answer": "Your basil might be getting too much water. Try letting the soil dry out a bit between waterings. Ensure it's getting plenty of sunlight too.",
    },
    "ask-cactus": {
        "user": "Question: How often should I water a cactus in winter?",
        "answer": "Reduce watering significantly in winter. Let the soil dry out completely between waterings, and only water sparingly when you do.",
    },
    "ask-tomato": {
        "user": "Plant: tomato. Question: Why do my tomato flowers fall off before fruit forms?",
        "answer": "This can happen if your tomato plant isn't getting enough sun, water, or nutrients. Extreme temperatures, both hot and cold, can also cause flowers to drop. Try to ensure consistent watering and good soil health.",
    },
}
BY_USER_MESSAGE = {r["user"]: r["answer"] for r in RECORDED.values()}
calls = []

async def replay_model(plant, question):
    """Replay: builds the same user message the recordings were made with and returns the recorded answer."""
    user = f"Plant: {plant}. Question: {question}" if plant else f"Question: {question}"
    calls.append(user)
    return BY_USER_MESSAGE[user]

call_model = replay_model

for name, r in RECORDED.items():
    print(f"{name}: {r['user']}")

ask-basil: Plant: basil. Question: The lower leaves on my basil are turning yellow. What should I do?
ask-cactus: Question: How often should I water a cactus in winter?
ask-tomato: Plant: tomato. Question: Why do my tomato flowers fall off before fruit forms?


## 5. The same contract in FastAPI

`Ask` is the request model. `Field(min_length=1, max_length=300)` sets the limits. `str | None = None` makes
plant optional: a field with a default is not required. `body: Ask` tells FastAPI to read the JSON body and check
it against `Ask` before the function runs. `-> Answer` is the reply's shape. Note that the route returns a dict
with a forgotten debugging field, `system_prompt`.

`send_all` posts the five requests to the app with `TestClient` and prints the status, whether the model was
called (did `calls` grow?) and the JSON that came back. (Starlette 1.10 prints a notice about its test client;
the first lines hide it.)

**Look for:** 200, 422, 200, 422, 200, and `model_called` is false for both 422s. Request 2: type `missing` at
`["body", "question"]`. Request 4: type `string_too_long` with `ctx` `{"max_length": 300}`. Each 200 has one
field, `answer`.

In [5]:
import warnings
warnings.filterwarnings("ignore", message=".*httpx2.*")
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, ConfigDict, Field

class Ask(BaseModel):
    question: str = Field(min_length=1, max_length=300)
    plant: str | None = None

class Answer(BaseModel):
    answer: str

app = FastAPI()

@app.post("/ask")
async def ask(body: Ask) -> Answer:
    text = await call_model(body.plant, body.question)
    return {"answer": text, "system_prompt": SYSTEM}

def send_all(app, short=False):
    client = TestClient(app)
    for r in tablet_requests:
        before = len(calls)
        res = client.post("/ask", json=r["body"])
        out = res.json()
        if short:  # one line per request: the answer's start, or each error's type and place
            out = out["answer"][:40] + " ..." if res.status_code == 200 else [(d["type"], d["loc"]) for d in out["detail"]]
        print(json.dumps({"name": r["name"], "status": res.status_code, "model_called": len(calls) > before, "json": out}))

send_all(app)
print("model calls so far:", len(calls))

{"name": "basil", "status": 200, "model_called": true, "json": {"answer": "Your basil might be getting too much water. Try letting the soil dry out a bit between waterings. Ensure it's getting plenty of sunlight too."}}
{"name": "fern-old-app", "status": 422, "model_called": false, "json": {"detail": [{"type": "missing", "loc": ["body", "question"], "msg": "Field required", "input": {"plant": "fern", "text": "Can I keep a fern in a bathroom with no window?"}}]}}
{"name": "cactus", "status": 200, "model_called": true, "json": {"answer": "Reduce watering significantly in winter. Let the soil dry out completely between waterings, and only water sparingly when you do."}}
{"name": "rose-long", "status": 422, "model_called": false, "json": {"detail": [{"type": "string_too_long", "loc": ["body", "question"], "msg": "String should have at most 300 characters", "input": "My rose bush was planted three years ago by the previous owner of the house, next to a wooden fence that gets sun in the morn

## 6. The response model filters what leaves

The same route twice for the basil request: once with `-> Answer`, once without a return type (the teammate who
deleted it "to keep it simple").

**Look for:** with `-> Answer` the tablet gets only `answer`. Without it, the response also carries
`system_prompt`: your instructions go to every tablet.

In [6]:
loose = FastAPI()

@loose.post("/ask")
async def ask_loose(body: Ask):
    text = await call_model(body.plant, body.question)
    return {"answer": text, "system_prompt": SYSTEM}

basil = tablet_requests[0]["body"]
print("with -> Answer:", json.dumps(TestClient(app).post("/ask", json=basil).json()))
print("without:       ", json.dumps(TestClient(loose).post("/ask", json=basil).json()))

with -> Answer: {"answer": "Your basil might be getting too much water. Try letting the soil dry out a bit between waterings. Ensure it's getting plenty of sunlight too."}
without:        {"answer": "Your basil might be getting too much water. Try letting the soil dry out a bit between waterings. Ensure it's getting plenty of sunlight too.", "system_prompt": "You answer plant-care questions from customers at a garden centre's shop tablet. Answer in at most three short sentences, in plain words. If the question is not about plants or gardening, say that you can only help with plants."}


## 7. The model call fails, with no handling

From here `call_model` is a real network call with httpx, to a local address where nothing listens (port 9),
so it fails for real. The route `bare` has no `try`. The cell sends the basil request twice: once as the tablet
sees it, and once with `TestClient` set to raise the server's exception, so you see what the server's log would show.

**Look for:** status **500**, content type `text/plain; charset=utf-8`, the text `Internal Server Error`. The
reason behind it: `httpx.ConnectError: All connection attempts failed`.

In [7]:
import httpx

async def call_model(plant, question):
    async with httpx.AsyncClient(timeout=5) as http:
        r = await http.post("http://127.0.0.1:9/v1/chat/completions", json={"question": question})
        return r.json()["answer"]

bare = FastAPI()

@bare.post("/ask")
async def ask_bare(body: Ask) -> Answer:
    text = await call_model(body.plant, body.question)
    return {"answer": text}

res = TestClient(bare, raise_server_exceptions=False).post("/ask", json=basil)
print("tablet gets:", res.status_code, "|", res.headers["content-type"], "|", res.text)
try:
    TestClient(bare).post("/ask", json=basil)
except Exception as e:
    print("server log: ", type(e).__module__ + "." + type(e).__name__ + ":", e)

tablet gets: 500 | text/plain; charset=utf-8 | Internal Server Error
server log:  httpx.ConnectError: All connection attempts failed


## 8. Catch it at the model call: a clean 502

The guarded route catches `httpx.HTTPError` around the model call, logs the reason with `log.exception`, and
raises `HTTPException(status_code=502, ...)`. FastAPI sends its `detail` as JSON. `log` is a logger from
Python's `logging` module. To keep the output short, its formatter prints only the last line of the traceback.

**Look for:** a log line `ERROR plant_helper: model call failed` ending with
`httpx.ConnectError: All connection attempts failed`. Then status **502**, content type `application/json`, and
`{"detail": "The plant helper cannot reach the model. Try again in a minute."}`.

In [8]:
import logging, sys, traceback

class LastLineOfTraceback(logging.Formatter):
    def formatException(self, ei):
        return "".join(traceback.format_exception_only(ei[1])).strip()

log = logging.getLogger("plant_helper")
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(LastLineOfTraceback("%(levelname)s %(name)s: %(message)s"))
log.handlers = [handler]
log.propagate = False

guarded = FastAPI()

@guarded.post("/ask")
async def ask_guarded(body: Ask) -> Answer:
    try:
        text = await call_model(body.plant, body.question)
    except httpx.HTTPError:
        log.exception("model call failed")
        raise HTTPException(status_code=502, detail="The plant helper cannot reach the model. Try again in a minute.")
    return {"answer": text}

res = TestClient(guarded).post("/ask", json=basil)
print("tablet gets:", res.status_code, "|", res.headers["content-type"], "|", json.dumps(res.json()))

ERROR plant_helper: model call failed
httpx.ConnectError: All connection attempts failed


tablet gets: 502 | application/json | {"detail": "The plant helper cannot reach the model. Try again in a minute."}


## Now change it

Three small experiments on the request model. Each cell has one line marked `# <- change this`. Change it and
run the cell again. The next cell puts the recorded answers back in place of the failing network call.

In [9]:
call_model = replay_model
print("call_model is the replay again")

call_model is the replay again


### Experiment 1: a tighter limit

The cell builds the app again with a `max_length` you choose and sends the five requests (one short line each).

**What should change:** with 300 you get the same results as in step 5. With 50, basil (66 characters) and tomato
(53) also get 422 `string_too_long`, and only the cactus question (44) reaches the model. The limit is checked
before the model call, so a rejected question costs nothing.

In [10]:
class Ask(BaseModel):
    question: str = Field(min_length=1, max_length=300)  # <- change this (try max_length=50)
    plant: str | None = None

app = FastAPI()

@app.post("/ask")
async def ask(body: Ask) -> Answer:
    text = await call_model(body.plant, body.question)
    return {"answer": text}

send_all(app, short=True)

{"name": "basil", "status": 200, "model_called": true, "json": "Your basil might be getting too much wat ..."}
{"name": "fern-old-app", "status": 422, "model_called": false, "json": [["missing", ["body", "question"]]]}
{"name": "cactus", "status": 200, "model_called": true, "json": "Reduce watering significantly in winter. ..."}
{"name": "rose-long", "status": 422, "model_called": false, "json": [["string_too_long", ["body", "question"]]]}
{"name": "tomato-extra", "status": 200, "model_called": true, "json": "This can happen if your tomato plant isn ..."}


### Experiment 2: make plant required

**What should change:** remove `| None = None` so the line reads `plant: str`. A field with no default is
required, so the cactus request, which has no plant, now gets 422 `missing` at `["body", "plant"]`, and only
basil and tomato reach the model. This is the kind of change that breaks callers who were fine yesterday.

In [11]:
class Ask(BaseModel):
    question: str = Field(min_length=1, max_length=300)
    plant: str | None = None  # <- change this (try: plant: str)

app = FastAPI()

@app.post("/ask")
async def ask(body: Ask) -> Answer:
    text = await call_model(body.plant, body.question)
    return {"answer": text}

send_all(app, short=True)

{"name": "basil", "status": 200, "model_called": true, "json": "Your basil might be getting too much wat ..."}
{"name": "fern-old-app", "status": 422, "model_called": false, "json": [["missing", ["body", "question"]]]}
{"name": "cactus", "status": 200, "model_called": true, "json": "Reduce watering significantly in winter. ..."}
{"name": "rose-long", "status": 422, "model_called": false, "json": [["string_too_long", ["body", "question"]]]}
{"name": "tomato-extra", "status": 200, "model_called": true, "json": "This can happen if your tomato plant isn ..."}


### Experiment 3: forbid unknown fields

By default Pydantic ignores fields it does not know (`extra="ignore"`). Set it to `"forbid"`.

**What should change:** the tomato request now gets 422 `extra_forbidden` at `["body", "tablet_id"]` instead of
an answer. Request 2 gets two errors: `missing` for question and `extra_forbidden` for `text`. Stricter is not
always better: an old tablet that sends one extra field would now get no answer at all.

In [12]:
class Ask(BaseModel):
    model_config = ConfigDict(extra="ignore")  # <- change this (try extra="forbid")
    question: str = Field(min_length=1, max_length=300)
    plant: str | None = None

app = FastAPI()

@app.post("/ask")
async def ask(body: Ask) -> Answer:
    text = await call_model(body.plant, body.question)
    return {"answer": text}

send_all(app, short=True)

{"name": "basil", "status": 200, "model_called": true, "json": "Your basil might be getting too much wat ..."}
{"name": "fern-old-app", "status": 422, "model_called": false, "json": [["missing", ["body", "question"]]]}
{"name": "cactus", "status": 200, "model_called": true, "json": "Reduce watering significantly in winter. ..."}
{"name": "rose-long", "status": 422, "model_called": false, "json": [["string_too_long", ["body", "question"]]]}
{"name": "tomato-extra", "status": 200, "model_called": true, "json": "This can happen if your tomato plant isn ..."}


## What you should see

- Step 5: requests 1, 3 and 5 get 200, and each response has one field, `answer`.
- Step 5: request 2 gets 422 with type `missing` at `["body", "question"]`, and `model_called` is false.
- Step 5: request 4 gets 422 with type `string_too_long` and `ctx` `{"max_length": 300}`.
- Step 6: without `-> Answer`, the basil response also carries `system_prompt`.
- Steps 7 and 8: with the model unreachable, the route without the except returns 500 and the text
  "Internal Server Error"; the guarded route returns 502 with a JSON `detail`.